# Session 1: The Loaded Die

**Computational Thinking · Habib University**

---

### Where we are going

By the end of this week you will have built a program that writes English
sentences. It will not be intelligent. It will not understand a word it says.
It will be a **die with a very large number of faces**, rolled over and over.

That is not a joke or a simplification for beginners. It is, to a first
approximation, what the autocomplete on your phone is doing, and it is a
recognisable ancestor of what ChatGPT is doing.

### Today

| Step | What we build | Idea |
|---|---|---|
| 1 | A fair die | random choice |
| 2 | A loaded die | outcomes carry **weights** |
| 3 | A die whose faces are words | the faces can be anything |
| 4 | A die with a bigger vocabulary | vocabulary size is a parameter |
| 5 | Sentences | repeat and join |
| 6 | Where do good weights come from? | we need **data** |

**Rules for this whole case study.** No `pandas`, no `numpy`, no `sklearn`.
We open files with `open()` and we process them with loops, lists and
dictionaries. Where something interesting happens, we write it ourselves,
because the point is to understand it.

---
## 0. The hook — your phone already does this

Open the keyboard on your phone. Type three words and look at the suggestion
bar. Type whatever it suggests, then whatever it suggests next, ten times over.

**Big question — hold on to this one, we come back to it on the last day:**

> Is your phone *understanding* you, or *counting* you?

Write your answer down now, in one sentence, before you know anything. 

---
## 1. A fair die

We start from working code that nobody can be intimidated by. No need to change anything.

`random.randint(a, b)` gives a whole number from `a` to `b`, ends included,
each equally likely.

In [1]:
import random

def roll_die():
    """Return a whole number from 1 to 6. Every face equally likely."""
    return random.randint(1, 6)


# Roll it ten times.
for i in range(10):
    print(roll_die(), end=" ")

2 5 1 6 4 1 5 2 3 1 

### Tallying in Python

To check whether our die is fair we need to count outcomes. A dictionary is
the natural shape: **the thing** on the left, **how many times we saw it** on
the right.

The line `dict[x] = counts.get(x, 0) + 1` is worth understanding for the question. 
`dict.get(x, 0)` means *"get the value for x key, or 0 if we have never
seen it"*. It is the idiom we will use for the rest of the week.

In [2]:
def tally(outcomes):
    """Count how often each outcome appears. Returns {outcome: count}.
    
    YOUR TASK
    ---------
    Build and return a dictionary whose keys are the outcomes and whose
    values are how many times each one appeared.

    Steps:
      1. Start with an empty dictionary.
      2. Loop over every item in `outcomes`.
      3. For each item, add 1 to its entry in the dictionary. The item may
         not be in the dictionary yet, so you cannot simply write
         `counts[face] = counts[face] + 1` -- that raises a KeyError the
         first time. Use `counts.get(face, 0)`, which gives you the current
         count, or 0 if this outcome has never been seen.
      4. Return the dictionary.
    """
    
    dict ={}
    for x in outcomes:
        dict[x] = dict.get(x, 0) + 1
        
    

    return dict


rolls = []
for i in range(600):
    rolls.append(roll_die())

print(tally(rolls))
# Expect roughly 100 of each face. "Roughly" is doing real work in that
# sentence -- run the cell a few times and watch the numbers move.

{5: 97, 6: 91, 4: 94, 3: 107, 2: 115, 1: 96}


**Discussion.** You expected 100 of each and you did not get 100 of each.
Is the die broken? How far from 100 would a count have to be before you
suspected the die rather than luck? Keep that instinct — in Session 3 we
measure things and have to decide when a difference is real.

---
## 2. A loaded die

Now a die where the faces are **not** equally likely — a six comes up half
the time.

To do that we stop thinking of a die as six faces and start thinking of it as
six **weights**:

```
fair:    {1: 1, 2: 1, 3: 1, 4: 1, 5: 1, 6: 1}
loaded:  {1: 1, 2: 1, 3: 1, 4: 1, 5: 1, 6: 5}
```

The loaded die has ten units of weight in total, five of which belong to the
face `6`. So a six comes up 5/10 of the time and every other face 1/10.

### How do you actually roll a weighted die?

Here is the picture. Lay the weights end to end along a ruler:

```
      1     2     3     4     5             6
   |-----|-----|-----|-----|-----|-----------------------|
   0     1     2     3     4     5                      10
```

Every face gets a stretch of ruler as long as its weight. Now throw a dart at
a uniformly random point between 0 and 10 and see which stretch it lands in.
Face `6` owns half the ruler, so it catches half the darts.

Turning the picture into code: walk along the faces adding up weights as you
go, and stop at the first face where the running total has passed the dart.

**We write this ourselves.** Python has a library function that does it
(`random.choices`), and we are not going to use it. The weighted pick *is*
the case study — if it is a library call, there is nothing here to learn.

**Predict first:** before running the cell, write down the tally you expect
from 100 rolls of the loaded die.

In [ ]:
def weighted_pick(weights):
    """Pick one key from `weights`, in proportion to its value.

    `weights` is a dictionary {outcome: weight}. Weights do not have to add
    up to anything in particular -- only their relative sizes matter.

    YOUR TASK
    ---------
    Implement the ruler-and-dart procedure from the cell above. This is the
    single most important function in the whole case study; every later step
    calls it. Take your time.

      1. Add up all the weights. Call it `total` -- the length of the ruler.
      2. Throw the dart: `target = random.random() * total`.
         `random.random()` returns a float from 0.0 up to (not including) 1.0,
         so `target` is a uniformly random point on the ruler.
      3. Walk through the outcomes keeping a `running` total of the weights
         you have passed so far. As soon as `running` is greater than
         `target`, the dart landed in this outcome's stretch -- return it.
      4. After the loop, return the last outcome anyway. You will essentially
         never reach this line; it is there because floating-point addition
         is not exact and the running total can end a hair below `total`.

    Check yourself: with weights {1: 1, 6: 3}, face 6 should come up about
    three times as often as face 1.
    """
    # your code here
    total = sum(weights.values())
    dicttotal = 0
    target = random.random()*total
    for i in weights:
        dicttotal+=weights[i]
        if target< dicttotal: 
            return i 

   
        

FAIR   = {1: 1, 2: 1, 3: 1, 4: 1, 5: 1, 6: 1}
LOADED = {1: 1, 2: 1, 3: 1, 4: 1, 5: 1, 6: 5}

rolls = []
for i in range(100):
    rolls.append(weighted_pick(LOADED))

print(tally(rolls))

{1: 4, 4: 8, 6: 54, 2: 17, 5: 8, 3: 9}


### Compare your prediction with what happened

Now run it again with far more rolls. Watch the proportions settle down.

In [4]:
for n in [100, 1000, 100000]:
    rolls = []
    for i in range(n):
        rolls.append(weighted_pick(LOADED))
    counts = tally(rolls)
    print("n =", n)
    for face in [1, 2, 3, 4, 5, 6]:
        share = counts.get(face, 0) / n
        print("   face", face, "->", round(100 * share, 2), "%")

n = 100
   face 1 -> 17.0 %
   face 2 -> 7.0 %
   face 3 -> 16.0 %
   face 4 -> 11.0 %
   face 5 -> 5.0 %
   face 6 -> 44.0 %
n = 1000
   face 1 -> 8.4 %
   face 2 -> 10.1 %
   face 3 -> 10.2 %
   face 4 -> 10.3 %
   face 5 -> 10.8 %
   face 6 -> 50.2 %
n = 100000
   face 1 -> 10.01 %
   face 2 -> 9.93 %
   face 3 -> 10.1 %
   face 4 -> 9.89 %
   face 5 -> 9.95 %
   face 6 -> 50.12 %


**Discussion.** At 100 rolls the numbers are all over the place; at 100,000
they are within a whisker of 10% / 10% / 10% / 10% / 10% / 50%.

The weights were exactly the same in both runs. What changed was how much
evidence you collected about them. Remember this on Friday when we ask why
language models are trained on enormous amounts of text.

---
## 3. The big idea — what if the faces are words?

Look carefully at `weighted_pick`. Nothing in it says *number*. It adds up
values, throws a dart, and returns a **key**. The keys could be anything.

So let us make the keys words.

In [5]:
SIX_WORDS = {"the": 1, "cat": 1, "sat": 1, "on": 1, "a": 1, "mat": 1}

for i in range(10):
    print(weighted_pick(SIX_WORDS), end=" ")

a mat cat cat mat a mat sat a on 

**Pause here.** We did not write a word generator. We wrote a die, and then
handed it words instead of numbers, and it became a word generator. Not one
line of `weighted_pick` changed.

A word generator and a loaded die are the same object. Everything else this
week is that sentence, applied harder.

### Brainstorm

What else could the faces be? Notes in a melody? Moves in a chess game?
Colours in the next pixel of an image? Amino acids?

All of those exist as real systems, and all of them are built out of exactly
this. Where do you think the idea starts to break down?

---
## 4. A bigger die

Six words is not much of a vocabulary. Let us go to a few dozen.

Notice what this costs us: nothing. Vocabulary size is just how many keys are
in the dictionary. A die with 50,000 faces is still a die — and 50,000 is
roughly the vocabulary size of a real language model's tokenizer.

In [6]:
VOCAB = {}

# Read the file "20Words.txt", extract individual words and make a dictionary, every word gets equal weight (1)
file = open("20words.txt")
for i in range(24):
    data =file.readline().strip()
    VOCAB[data] = VOCAB.get(data, 0) + 1

print("vocabulary size:", len(VOCAB))
for i in range(12):
    print(weighted_pick(VOCAB), end=" ")

vocabulary size: 24
is maybe about my the and that and hi change hi that 

---
## 5. From words to sentences

Roll the word die repeatedly and stick the results together. That is a text
generator. It is a bad one, and its badness is the reason the rest of the week
exists.

In [7]:
def make_sentence(weights, length):
    """Roll the word die `length` times and join the results with spaces.

    YOUR TASK
    ---------
      1. Build a list by calling `weighted_pick(weights)` `length` times.
      2. Turn that list into one string, with a single space between words
         and no space at either end. Build it with a loop and `+=` rather
         than reaching for a library -- you want to be able to see exactly
         where the spaces come from.

    Return the string.
    """
    # your code here
    list=[]
    for i in range(length):
        num = weighted_pick(weights)
        list.append(num)
    str1=""
    for i in list:
        str1+=i+" "
    return str1[:-1]
for i in range(5):
    print(make_sentence(VOCAB, 12))

about it do maybe topic the how do change is my else
do change topic talk do name maybe is change hi my we
name change hi not remember topic do and the and we name
about I talk it was how topic do I topic about something
about was talk know was and do is the maybe do we
